# Lesson 2.5 - File I/O and Error Handling

**Objectives**

- Read from and write to text and CSV files with Python's built-ins
- Explain what exceptions are and why programs raise them
- Handle errors gracefully with `try` / `except` / `finally`

See `notes.md` (Lesson 2.5) for the full written explanation.

This notebook actually creates files on disk as you run it. To keep the project folder tidy, everything gets written into a `scratch/` subfolder next to this notebook - that folder is just for this lesson's exercise output, not part of the course materials themselves.

In [1]:
import os

os.makedirs("scratch", exist_ok=True)
print("scratch/ is ready")

scratch/ is ready


## Reading and writing text files

In [2]:
with open("scratch/notes.txt", "w") as file:
    file.write("Hello, file!\n")
    file.write("This is line two.\n")

print("wrote notes.txt")

wrote notes.txt


In [3]:
with open("scratch/notes.txt", "r") as file:
    contents = file.read()

print(contents)

Hello, file!
This is line two.



In [4]:
# "a" (append) adds to the file WITHOUT erasing what's already there
with open("scratch/notes.txt", "a") as file:
    file.write("This is line three.\n")

with open("scratch/notes.txt", "r") as file:
    for line in file:
        print(line.strip())   # .strip() removes the trailing newline

Hello, file!
This is line two.
This is line three.


## Reading and writing CSV files

In [5]:
import csv

rows = [
    ["name", "score"],
    ["Ada", 95],
    ["Grace", 88],
]

with open("scratch/scores.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("wrote scores.csv")

wrote scores.csv


In [6]:
with open("scratch/scores.csv", "r", newline="") as file:
    reader = csv.DictReader(file)
    for row in reader:
        print(row)

{'name': 'Ada', 'score': '95'}
{'name': 'Grace', 'score': '88'}


Notice the scores came back as **strings** (`'95'`, not `95`) - every value read from a CSV file starts out as text. You'd need `int(row["score"])` to do arithmetic with it. In Module 3, `pandas` handles this conversion for you automatically.

### A quick peek at a real CSV file

The course's shared dataset includes `data/raw/products.csv`. Let's read just the first few rows with the plain `csv` module, before we ever touch pandas in Module 3 - so you can see what pandas will be doing for you under the hood.

In [7]:
import os

products_path = os.path.join("..", "..", "..", "data", "raw", "products.csv")

if os.path.exists(products_path):
    with open(products_path, "r", newline="") as file:
        reader = csv.DictReader(file)
        for i, row in enumerate(reader):
            if i >= 3:
                break
            print(row["product_name"], "-", row["price"])
else:
    print("products.csv not found at the expected path - skipping this optional peek.")

Mini Headphones - 409.02
Classic Throw Pillow - 292.93
Premium Shampoo - 38.78


## What is an exception?

In [8]:
try:
    print(10 / 0)
except ZeroDivisionError as error:
    print(f"Caught an error: {error}")

Caught an error: division by zero


An **exception** is Python's way of signaling something went wrong. Left uncaught, it stops the program and prints a **traceback**. We already caught it above with `try` / `except` instead of letting it crash.

## `try` / `except` / `finally`

In [9]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Error: cannot divide by zero.")
        return None
    except TypeError as error:
        print(f"Error: {error}")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(10, "2"))

5.0
Error: cannot divide by zero.
None
Error: unsupported operand type(s) for /: 'int' and 'str'
None


In [10]:
try:
    file = open("scratch/notes.txt", "r")
    contents = file.read()
finally:
    file.close()
    print("File closed.")

File closed.


`finally` runs **no matter what** - whether the `try` block succeeded or an exception was caught. In practice, `with` (used earlier in this notebook) already handles closing files for you, so you'll reach for explicit `finally` blocks less often - but it's good to recognize the pattern.

In [11]:
try:
    with open("scratch/does_not_exist.txt", "r") as file:
        contents = file.read()
except FileNotFoundError:
    print("That file doesn't exist yet.")

That file doesn't exist yet.


> **Beginner mistake: catching every exception with a bare `except:`.** This hides real bugs along with the errors you meant to catch. Always name the specific exception(s) you expect, like `except FileNotFoundError:`.

## Try it yourself

**Exercise 1.** Write a list of 3 short strings representing a to-do list. Write them to `scratch/todo.txt`, one per line (use a `for` loop and `file.write(item + "\n")`).

In [12]:
# TODO: write a 3-item to-do list to scratch/todo.txt, one item per line



**Exercise 2.** Read `scratch/todo.txt` back and print each line with its line number, like `"1. Buy milk"` (use `enumerate(file, start=1)`).

In [13]:
# TODO: read scratch/todo.txt and print each line with its number



**Exercise 3.** Write a function `safe_get_item(items, index)` that returns `items[index]`, but catches `IndexError` and returns `None` instead of crashing. Test it with a valid index and an out-of-range index.

In [14]:
# TODO: define safe_get_item(items, index) with try/except IndexError



### Solution

In [15]:
# Exercise 1
todo_items = ["Buy milk", "Write notes", "Walk the dog"]
with open("scratch/todo.txt", "w") as file:
    for item in todo_items:
        file.write(item + "\n")

# Exercise 2
with open("scratch/todo.txt", "r") as file:
    for line_number, line in enumerate(file, start=1):
        print(f"{line_number}. {line.strip()}")

# Exercise 3
def safe_get_item(items, index):
    try:
        return items[index]
    except IndexError:
        return None

print(safe_get_item([1, 2, 3], 1))
print(safe_get_item([1, 2, 3], 10))

1. Buy milk
2. Write notes
3. Walk the dog
2
None
